In [1]:
import os
import pandas as pd

BASE_DIR=r"E:\Rasengan\Cloud-Removal"

METADATA_DIR=os.path.join(
    BASE_DIR,
    "metadata"
)

PATCH_DIR=os.path.join(
    BASE_DIR,
    "data",
    "patches"
)

CSV_FILES = {
    "patches1": os.path.join(METADATA_DIR, "patch1.csv"),
    "patches2": os.path.join(METADATA_DIR, "patch2.csv"),
    "patches3": os.path.join(METADATA_DIR, "patch3.csv")
}

In [4]:
required_columns = [
    "pair_id",
    "patch_id",
    "cloudy_filename",
    "noncloudy_filename",
    "row",
    "col",
    "patch_size",
    "stride",
    "cloudy_valid_pct",
    "noncloudy_valid_pct"
]


dataframes = []

for source, csv_path in CSV_FILES.items():

    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f"CSV not found: {csv_path}"
        )

    df = pd.read_csv(csv_path)

    missing = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{source} is missing columns: {missing}"
        )

    df["source"] = source

    df["group_id"] = (
        df["source"].astype(str)
        + "_"
        + df["pair_id"].astype(str)
    )

    dataframes.append(df)

    print(
        f"{source}: "
        f"{len(df):,} patches"
    )


master_df = pd.concat(
    dataframes,
    ignore_index=True
)


patches1: 100,000 patches
patches2: 65,000 patches
patches3: 70,001 patches


In [5]:
print("\n" + "=" * 50)
print("MASTER DATASET")
print("=" * 50)

print(
    "Total patches:",
    f"{len(master_df):,}"
)

print(
    "Unique groups:",
    master_df["group_id"].nunique()
)

print(
    "Unique pair IDs:",
    master_df["pair_id"].nunique()
)

print("\nPatches by source:")
print(
    master_df["source"].value_counts()
)

print("\nColumns:")
print(
    master_df.columns.tolist()
)

print("\nFirst 5 rows:")
display(
    master_df.head()
)


MASTER DATASET
Total patches: 235,001
Unique groups: 48
Unique pair IDs: 48

Patches by source:
source
patches1    100000
patches3     70001
patches2     65000
Name: count, dtype: int64

Columns:
['pair_id', 'patch_id', 'cloudy_filename', 'noncloudy_filename', 'row', 'col', 'patch_size', 'stride', 'cloudy_valid_pct', 'noncloudy_valid_pct', 'source', 'group_id']

First 5 rows:


,pair_id,patch_id,cloudy_filename,noncloudy_filename,row,col,patch_size,stride,cloudy_valid_pct,noncloudy_valid_pct,source,group_id
0,ahemadabad1,ahemadabad1_patch_00001,ahemadabad1_patch_00001.png,ahemadabad1_patch_00001.png,15872,1920,256,128,100.0,100.0,patches1,patches1_ahemadabad1
1,ahemadabad1,ahemadabad1_patch_00002,ahemadabad1_patch_00002.png,ahemadabad1_patch_00002.png,12416,2560,256,128,100.0,100.0,patches1,patches1_ahemadabad1
2,ahemadabad1,ahemadabad1_patch_00003,ahemadabad1_patch_00003.png,ahemadabad1_patch_00003.png,13568,1664,256,128,100.0,100.0,patches1,patches1_ahemadabad1
3,ahemadabad1,ahemadabad1_patch_00004,ahemadabad1_patch_00004.png,ahemadabad1_patch_00004.png,14464,13696,256,128,100.0,100.0,patches1,patches1_ahemadabad1
4,ahemadabad1,ahemadabad1_patch_00005,ahemadabad1_patch_00005.png,ahemadabad1_patch_00005.png,14080,10368,256,128,100.0,100.0,patches1,patches1_ahemadabad1


In [6]:
missing_cloudy = []
missing_noncloudy = []

for _, row in master_df.iterrows():

    source = row["source"]

    cloudy_path = os.path.join(
        PATCH_DIR,
        source,
        "cloudy",
        row["cloudy_filename"]
    )

    noncloudy_path = os.path.join(
        PATCH_DIR,
        source,
        "non-cloudy",
        row["noncloudy_filename"]
    )

    if not os.path.isfile(cloudy_path):
        missing_cloudy.append(cloudy_path)

    if not os.path.isfile(noncloudy_path):
        missing_noncloudy.append(noncloudy_path)


print("=" * 60)
print("DATASET FILE CHECK")
print("=" * 60)

print(
    "Total records:",
    f"{len(master_df):,}"
)

print(
    "Missing cloudy images:",
    len(missing_cloudy)
)

print(
    "Missing non-cloudy images:",
    len(missing_noncloudy)
)

if missing_cloudy:
    print("\nFirst missing cloudy files:")
    for path in missing_cloudy[:10]:
        print(path)

if missing_noncloudy:
    print("\nFirst missing non-cloudy files:")
    for path in missing_noncloudy[:10]:
        print(path)

if not missing_cloudy and not missing_noncloudy:
    print("\n✓ Every cloudy/non-cloudy image exists.")
else:
    print("\n⚠ Missing files detected. Do not continue to splitting yet.")

DATASET FILE CHECK
Total records: 235,001
Missing cloudy images: 0
Missing non-cloudy images: 0

✓ Every cloudy/non-cloudy image exists.


In [7]:
print("=" * 60)
print("METADATA QUALITY CHECK")
print("=" * 60)

duplicate_patch_ids = master_df[
    master_df["patch_id"].duplicated(keep=False)
]

duplicate_rows = master_df[
    master_df.duplicated(
        subset=["source", "patch_id"],
        keep=False
    )
]

invalid_patch_size = master_df[
    master_df["patch_size"] != 256
]

invalid_stride = master_df[
    master_df["stride"] != 128
]

invalid_cloudy_valid = master_df[
    master_df["cloudy_valid_pct"] < 90
]

invalid_noncloudy_valid = master_df[
    master_df["noncloudy_valid_pct"] < 90
]

print(
    "Duplicate patch IDs:",
    len(duplicate_patch_ids)
)

print(
    "Duplicate source + patch IDs:",
    len(duplicate_rows)
)

print(
    "Invalid patch size:",
    len(invalid_patch_size)
)

print(
    "Invalid stride:",
    len(invalid_stride)
)

print(
    "Cloudy valid < 90%:",
    len(invalid_cloudy_valid)
)

print(
    "Non-cloudy valid < 90%:",
    len(invalid_noncloudy_valid)
)

print("\nUnique sources:")
print(
    master_df["source"].value_counts()
)

print("\nUnique groups:")
print(
    master_df["group_id"].nunique()
)

if (
    len(duplicate_rows) == 0
    and len(invalid_patch_size) == 0
    and len(invalid_stride) == 0
):
    print("\n✓ Metadata structure looks good.")
else:
    print("\n⚠ Metadata problems detected.")

METADATA QUALITY CHECK
Duplicate patch IDs: 0
Duplicate source + patch IDs: 0
Invalid patch size: 0
Invalid stride: 0
Cloudy valid < 90%: 0
Non-cloudy valid < 90%: 0

Unique sources:
source
patches1    100000
patches3     70001
patches2     65000
Name: count, dtype: int64

Unique groups:
48

✓ Metadata structure looks good.


In [8]:
from sklearn.model_selection import train_test_split

groups = master_df["group_id"].unique()

train_groups, temp_groups = train_test_split(
    groups,
    test_size=0.20,
    random_state=42
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=42
)

train_df = master_df[
    master_df["group_id"].isin(train_groups)
].copy()

val_df = master_df[
    master_df["group_id"].isin(val_groups)
].copy()

test_df = master_df[
    master_df["group_id"].isin(test_groups)
].copy()

print("=" * 60)
print("DATASET SPLIT")
print("=" * 60)

print(f"Total pairs : {len(master_df):,}")
print(f"Train pairs : {len(train_df):,}")
print(f"Val pairs   : {len(val_df):,}")
print(f"Test pairs  : {len(test_df):,}")

print("\nGroups:")
print(f"Train groups: {len(train_groups)}")
print(f"Val groups  : {len(val_groups)}")
print(f"Test groups : {len(test_groups)}")

print("\n✓ Scene-level split completed.")

DATASET SPLIT
Total pairs : 235,001
Train pairs : 185,001
Val pairs   : 25,000
Test pairs  : 25,000

Groups:
Train groups: 38
Val groups  : 5
Test groups : 5

✓ Scene-level split completed.


In [9]:
MASTER_CSV = os.path.join(METADATA_DIR, "master_metadata.csv")
TRAIN_CSV = os.path.join(METADATA_DIR, "train.csv")
VAL_CSV = os.path.join(METADATA_DIR, "val.csv")
TEST_CSV = os.path.join(METADATA_DIR, "test.csv")

master_df.to_csv(MASTER_CSV, index=False)
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

print("=" * 60)
print("METADATA FILES SAVED")
print("=" * 60)

print(f"Master : {MASTER_CSV}")
print(f"Train  : {TRAIN_CSV}")
print(f"Val    : {VAL_CSV}")
print(f"Test   : {TEST_CSV}")

print("\n✓ All metadata files saved successfully.")

METADATA FILES SAVED
Master : E:\Rasengan\Cloud-Removal\metadata\master_metadata.csv
Train  : E:\Rasengan\Cloud-Removal\metadata\train.csv
Val    : E:\Rasengan\Cloud-Removal\metadata\val.csv
Test   : E:\Rasengan\Cloud-Removal\metadata\test.csv

✓ All metadata files saved successfully.
